### 1. Install Julia

We download and extract the Julia 1.10.4 binary and set up a symbolic link so we can run julia from the command line.

In [1]:
%%shell
wget -q https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-1.10.4-linux-x86_64.tar.gz
tar -xzf julia-1.10.4-linux-x86_64.tar.gz
ln -sf /content/julia-1.10.4/bin/julia /usr/local/bin/julia
julia --version

julia version 1.10.4


In [ ]:
%%shell
julia -e 'using Pkg; Pkg.add(["CUDA", "BenchmarkTools", "Statistics"])'

### 2. Install Julia Packages
We install the `CUDA.jl` package for GPU acceleration, `BenchmarkTools.jl` for precise timing, and `Statistics.jl` for data analysis.

In [5]:
%%writefile bench-gpu.jl
using CUDA, BenchmarkTools, Statistics


Writing bench-gpu.jl


In [9]:
!touch bench-gpu.jl

### 3. Create the Benchmark Script
We write the Julia source code to a file named `bench-gpu.jl`. This script contains the CUDA kernel for pairwise distances and the benchmarking logic.

In [13]:
%%writefile bench-gpu.jl

using CUDA
using BenchmarkTools
using Statistics
using Dates

# ------------------------------------------------------------------------------
# GPU kernel
# Each CUDA thread computes one distance entry distances[i, j]
# ------------------------------------------------------------------------------
function distance_kernel!(distances, px, py, pz, n)
    i = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    j = (blockIdx().y - 1) * blockDim().y + threadIdx().y

    if i <= n && j <= n
        dx = px[i] - px[j]
        dy = py[i] - py[j]
        dz = pz[i] - pz[j]
        @inbounds distances[i, j] = sqrt(dx*dx + dy*dy + dz*dz)
    end
    return
end

# ------------------------------------------------------------------------------
# GPU pairwise distance wrapper
# ------------------------------------------------------------------------------
function pairwise_distances_gpu(points_cpu::Matrix{Float32})
    n = size(points_cpu, 1)

    px = CuArray(points_cpu[:, 1])
    py = CuArray(points_cpu[:, 2])
    pz = CuArray(points_cpu[:, 3])

    out = CUDA.zeros(Float32, n, n)

    threads = (16, 16)
    blocks = (cld(n, threads[1]), cld(n, threads[2]))

    @cuda threads=threads blocks=blocks distance_kernel!(out, px, py, pz, n)
    CUDA.synchronize()

    return out
end

# ------------------------------------------------------------------------------
# CSV writer
# ------------------------------------------------------------------------------
function append_result_to_csv(
    csv_path::String;
    device_name::String,
    compute_capability,
    vram_gb::Float64,
    n::Int,
    threads_mode::String,
    median_s::Float64,
    min_s::Float64,
    mean_s::Float64,
    throughput_ops_per_sec::Float64,
)
    if !isfile(csv_path)
        open(csv_path, "w") do io
            println(io,
                "device_name,compute_capability,vram_gb,n,threads_mode,median_s,min_s,mean_s,throughput_ops_per_sec,timestamp"
            )
        end
    end

    timestamp = Dates.format(now(), dateformat"yyyy-mm-ddTHH:MM:SS")

    open(csv_path, "a") do io
        println(io,
            string(
                device_name, ",",
                compute_capability, ",",
                round(vram_gb, digits=2), ",",
                n, ",",
                threads_mode, ",",
                round(median_s, digits=6), ",",
                round(min_s, digits=6), ",",
                round(mean_s, digits=6), ",",
                round(throughput_ops_per_sec, digits=2), ",",
                timestamp
            )
        )
    end
end

# ------------------------------------------------------------------------------
# Main benchmark
# ------------------------------------------------------------------------------
function main()
    if !CUDA.functional()
        error("CUDA is not functional on this system.")
    end

    dev = CUDA.device()
    device_name = CUDA.name(dev)
    compute_cap = CUDA.capability(dev)
    vram_gb = CUDA.totalmem(dev) / 1024^3

    println()
    println("=== GPU Device ===")
    println("Name            : ", device_name)
    println("VRAM            : ", round(vram_gb, digits=2), " GB")
    println("Compute cap     : ", compute_cap)
    println()

    n = 10_000
    points_cpu = rand(Float32, n, 3)

    println("=== Warm-up ===")
    warmup_pts = rand(Float32, 100, 3)
    pairwise_distances_gpu(warmup_pts)
    CUDA.synchronize()
    println("Warm-up done.")
    println()

    println("=== Benchmarking GPU kernel ===")
    println("Dataset size    : ", n, " points")
    println()

    result = @benchmark begin
        pairwise_distances_gpu($points_cpu)
        CUDA.synchronize()
    end samples=10 evals=1 seconds=300

    median_s = median(result).time * 1e-9
    min_s = minimum(result).time * 1e-9
    mean_s = mean(result).time * 1e-9
    throughput = Float64(n * n) / median_s

    println("============================================================")
    println("GPU BENCHMARK RESULTS")
    println("============================================================")
    println("GPU name         : ", device_name)
    println("Dataset          : n = ", n, " points (Float32)")
    println("Median time      : ", round(median_s * 1000, digits=3), " ms")
    println("Min time         : ", round(min_s * 1000, digits=3), " ms")
    println("Mean time        : ", round(mean_s * 1000, digits=3), " ms")
    println("Throughput       : ", round(throughput / 1e6, digits=2), " M ops/sec")
    println("============================================================")
    println()

    csv_path = "benchmark_results_gpu.csv"

    append_result_to_csv(
        csv_path;
        device_name=device_name,
        compute_capability=compute_cap,
        vram_gb=vram_gb,
        n=n,
        threads_mode="gpu",
        median_s=median_s,
        min_s=min_s,
        mean_s=mean_s,
        throughput_ops_per_sec=throughput,
    )

    println("CSV saved/appended to: ", abspath(csv_path))
end

main()


Overwriting bench-gpu.jl


### Run the Script

In [17]:
%%shell
julia bench-gpu.jl


=== GPU Device ===
Name            : Tesla T4
VRAM            : 14.56 GB
Compute cap     : 7.5.0

=== Warm-up ===
Warm-up done.

=== Benchmarking GPU kernel ===
Dataset size    : 10000 points

GPU BENCHMARK RESULTS
GPU name         : Tesla T4
Dataset          : n = 10000 points (Float32)
Median time      : 6.623 ms
Min time         : 6.355 ms
Mean time        : 6.608 ms
Throughput       : 15098.24 M ops/sec

CSV saved/appended to: /content/benchmark_results_gpu.csv
